 ### 规格化美国动物数据

In [98]:
# 一年一年处理
# 第一列肉牛
import pandas as pd
year = ["1997","2002","2007","2012","2017"]
for y in year:
    animal = pd.DataFrame()
    data_livestock = pd.read_csv('standard/county/动物类/livestock_standard/'+str(y)+'.csv')

    # Domain只要TOTAL
    data_livestock = data_livestock.loc[data_livestock["Domain"]=="TOTAL"].reset_index(drop=True)

    # 填写州县，每行一个
    animal = data_livestock.drop_duplicates(subset=["OBJECTID"]).loc[:,["OBJECTID","State","County"]].reset_index(drop=True)
    animal["year"] = y # 设定年份

    # 提取出肉牛的，使用左连接
    cattle_beef = data_livestock.loc[data_livestock["Data Item"]=="CATTLE, COWS, BEEF - INVENTORY"].reset_index(drop=True)
    animal = animal.merge(cattle_beef[["Value","OBJECTID"]],on="OBJECTID",how="left")
    animal = animal.rename(columns={"Value":"cattle_cow_beef"}) # 改名

    # 提取出奶牛的，使用左连接
    cattle_milk = data_livestock.loc[data_livestock["Data Item"]=="CATTLE, COWS, MILK - INVENTORY"].reset_index(drop=True)
    animal = animal.merge(cattle_milk[["Value","OBJECTID"]],on="OBJECTID",how="left")
    animal = animal.rename(columns={"Value":"cattle_cow_milk"}) # 改名

    # 提取出其它牛，使用左连接
    cattle_other = data_livestock.loc[data_livestock["Data Item"]=="CATTLE, (EXCL COWS) - INVENTORY"].reset_index(drop=True)
    animal = animal.merge(cattle_other[["Value","OBJECTID"]],on="OBJECTID",how="left")
    animal = animal.rename(columns={"Value":"cattle_excl_cows"}) # 改名

    # 提取猪，使用左连接
    hogs = data_livestock.loc[data_livestock["Data Item"]=="HOGS - INVENTORY"].reset_index(drop=True)
    animal = animal.merge(hogs[["Value","OBJECTID"]],on="OBJECTID",how="left")
    animal = animal.rename(columns={"Value":"hogs"}) # 改名

    # 提取羊和山羊,使用左连接
    # 提取Goats
    goats = data_livestock.loc[data_livestock["Data Item"]=="GOATS - INVENTORY"].reset_index(drop=True)
    animal = animal.merge(goats[["Value","OBJECTID"]],on="OBJECTID",how="left")
    animal = animal.rename(columns={"Value":"goats"}) # 改名

    # 提取Sheep_EWES
    sheep_ewes = data_livestock.loc[data_livestock["Data Item"]=="SHEEP, EWES, BREEDING, GE 1 YEAR - INVENTORY"].reset_index(drop=True)
    animal = animal.merge(sheep_ewes[["Value","OBJECTID"]],on="OBJECTID",how="left")
    animal = animal.rename(columns={"Value":"sheep_ewes"}) # 改名

    # 提取Sheep_LAMBS
    sheep_lambs = data_livestock.loc[data_livestock["Data Item"]=="SHEEP, INCL LAMBS - INVENTORY"].reset_index(drop=True)
    animal = animal.merge(sheep_lambs[["Value","OBJECTID"]],on="OBJECTID",how="left")
    animal = animal.rename(columns={"Value":"sheep_lambs"}) # 改名

    # 将这3种羊的Value相加形成新的一列羊
    def safe_numeric(x):
        try:
            return pd.to_numeric(x)
        except ValueError:
            return 0

    cols_to_convert = ["goats","sheep_ewes","sheep_lambs"]

    for col in cols_to_convert:
        animal[col] = animal[col].str.replace(',', '')  # remove commas

    animal["sheep_goats"] = animal[cols_to_convert].applymap(safe_numeric).sum(axis=1) # 新的一列

    # 删除这3列
    animal = animal.drop(columns=cols_to_convert)

    # 接下来处理禽类
    data_pultry = pd.read_csv('standard/county/动物类/poultry_standard/'+str(y)+'.csv') 

    # Domain只要TOTAL
    data_pultry = data_pultry.loc[data_pultry["Domain"]=="TOTAL"].reset_index(drop=True)

    # 提取出肉鸡，使用左连接
    broilers = data_pultry.loc[data_pultry["Data Item"]=="CHICKENS, BROILERS - INVENTORY"].reset_index(drop=True)
    animal = animal.merge(broilers[["Value","OBJECTID"]],on="OBJECTID",how="left")
    animal = animal.rename(columns={"Value":"broilers"}) # 改名

    # 提取出蛋鸡，使用左连接
    layers = data_pultry.loc[data_pultry["Data Item"]=="CHICKENS, LAYERS - INVENTORY"].reset_index(drop=True)
    animal = animal.merge(layers[["Value","OBJECTID"]],on="OBJECTID",how="left")
    animal = animal.rename(columns={"Value":"layers"}) # 改名

    # 添加马、驴、骡、兔子各一列，数据缺失，NAN代替
    animal["horses"] = None
    animal["donkeys"] = None
    animal["mules"] = None
    animal["rabbits"] = None

    # 计算肉牛的出栏量
    # 确定每个州的肉牛出栏总量
    state_slaughtered = pd.read_csv("standard/state/动物类/livestocks_standard/"+str(y)+".csv")
    # 取出牛贸易屠宰的，使用左连接
    state_slaughtered_cattle_beef_com = state_slaughtered.loc[state_slaughtered["Data Item"]=="CATTLE, CALVES, SLAUGHTER, COMMERCIAL - SLAUGHTERED, MEASURED IN HEAD"].reset_index(drop=True)
    animal = animal.merge(state_slaughtered_cattle_beef_com[["Value","State"]],on="State",how="left")
    animal = animal.rename(columns={"Value":"state_slaughtered_cattle_beef_com"}) # 改名

    # 取出牛农场屠宰的，使用左连接
    state_slaughtered_cattle_beef_farm = state_slaughtered.loc[state_slaughtered["Data Item"]=="CATTLE, INCL CALVES, SLAUGHTER, ON FARM - SLAUGHTERED, MEASURED IN HEAD"].reset_index(drop=True)
    animal = animal.merge(state_slaughtered_cattle_beef_farm[["Value","State"]],on="State",how="left")
    animal = animal.rename(columns={"Value":"state_slaughtered_cattle_beef_farm"}) # 改名

    # 将这2种牛的Value相加形成新的一列作为出栏总量
    cols_to_convert = ["state_slaughtered_cattle_beef_com","state_slaughtered_cattle_beef_farm"]

    for col in cols_to_convert:
        animal[col] = animal[col].str.replace(',', '')  # remove commas

    animal["cattle_slaughtered_sum"] = animal[cols_to_convert].applymap(safe_numeric).sum(axis=1) # 新的一列

    # 删除这2列
    animal = animal.drop(columns=cols_to_convert)

    # 计算肉牛在该州的存栏占比
    animal["cattle_cow_beef"] = animal["cattle_cow_beef"].str.replace(',', '')  # remove commas
    # 将该列转化成数值类型，转化不了的用0代替
    animal["cattle_cow_beef"] = pd.to_numeric(animal["cattle_cow_beef"], errors='coerce').fillna(0)
    # 计算每个州的肉牛存栏总数
    state_total = animal.groupby('State')['cattle_cow_beef'].transform('sum')
    # 计算每个县在其所在州的肉牛存栏占比
    animal["inventory_ratio"] = animal["cattle_cow_beef"] / state_total

    # 用该县所在州的出栏总和乘以该县的存栏占比，得到该县的肉牛出栏量
    animal["cattle_beef_slaughtered"] = animal["cattle_slaughtered_sum"] * animal["inventory_ratio"]

    # 去除出栏总量这一列
    animal = animal.drop(columns=["cattle_slaughtered_sum","inventory_ratio"])

    # 同样的方法计算猪的出栏量
    # 确定每个州的猪出栏总量
    # 取出猪贸易屠宰的，使用左连接
    state_slaughtered_hogs_com = state_slaughtered.loc[state_slaughtered["Data Item"]=="HOGS, SLAUGHTER, COMMERCIAL - SLAUGHTERED, MEASURED IN HEAD"].reset_index(drop=True)
    animal = animal.merge(state_slaughtered_hogs_com[["Value","State"]],on="State",how="left")
    animal = animal.rename(columns={"Value":"state_slaughtered_hogs_com"}) # 改名

    # 取出猪农场屠宰的，使用左连接
    state_slaughtered_hogs_farm = state_slaughtered.loc[state_slaughtered["Data Item"]=="HOGS, SLAUGHTER, ON FARM - SLAUGHTERED, MEASURED IN HEAD"].reset_index(drop=True)
    animal = animal.merge(state_slaughtered_hogs_farm[["Value","State"]],on="State",how="left")
    animal = animal.rename(columns={"Value":"state_slaughtered_hogs_farm"}) # 改名

    # 将这2种猪的Value相加形成新的一列作为出栏总量
    cols_to_convert = ["state_slaughtered_hogs_com","state_slaughtered_hogs_farm"]

    for col in cols_to_convert:
        animal[col] = animal[col].str.replace(',', '')  # remove commas

    animal["hogs_slaughtered_sum"] = animal[cols_to_convert].applymap(safe_numeric).sum(axis=1) # 新的一列

    # 删除这2列
    animal = animal.drop(columns=cols_to_convert)

    # 计算猪在该州的存栏占比
    animal["hogs"] = animal["hogs"].str.replace(',', '')  # remove commas
    # 将该列转化成数值类型，转化不了的用0代替
    animal["hogs"] = pd.to_numeric(animal["hogs"], errors='coerce').fillna(0)
    # 计算每个州的猪存栏总数
    state_total = animal.groupby('State')['hogs'].transform('sum')
    # 计算每个县在其所在州的猪存栏占比
    animal["inventory_ratio"] = animal["hogs"] / state_total

    # 用该县所在州的出栏总和乘以该县的存栏占比，得到该县的猪出栏量
    animal["hogs_slaughtered"] = animal["hogs_slaughtered_sum"] * animal["inventory_ratio"]

    # 去除出栏总量这一列
    animal = animal.drop(columns=["hogs_slaughtered_sum","inventory_ratio"])

    # 接下来计算肉鸡出栏量
    # 确定每个州的肉鸡出栏总量
    state_slaughtered = pd.read_csv("standard/state/动物类/poultry_standard/"+str(y)+".csv")

    # 取出肉鸡屠宰的，使用左连接
    state_slaughtered_broilers = state_slaughtered.loc[state_slaughtered["Data Item"]=="CHICKENS, YOUNG, SLAUGHTER, FI - SLAUGHTERED, MEASURED IN HEAD"].reset_index(drop=True)
    animal = animal.merge(state_slaughtered_broilers[["Value","State"]],on="State",how="left")
    animal = animal.rename(columns={"Value":"state_slaughtered_broilers"}) # 改名

    # 将该列转化成数值类型，转化不了的用0代替
    animal["state_slaughtered_broilers"] = animal["state_slaughtered_broilers"].str.replace(',', '')  # remove commas
    animal["state_slaughtered_broilers"] = pd.to_numeric(animal["state_slaughtered_broilers"], errors='coerce').fillna(0)

    # 将该列转化成数值类型，转化不了的用0代替
    animal["broilers"] = animal["broilers"].str.replace(',', '')  # remove commas
    animal["broilers"] = pd.to_numeric(animal["broilers"], errors='coerce').fillna(0)

    # 计算每个州的肉鸡存栏总数
    state_total = animal.groupby('State')['broilers'].transform('sum')
    # 计算每个县在其所在州的肉鸡存栏占比
    animal["inventory_ratio"] = animal["broilers"] / state_total

    # 用该县所在州的出栏总和乘以该县的存栏占比，得到该县的肉鸡出栏量
    animal["broilers_slaughtered"] = animal["state_slaughtered_broilers"] * animal["inventory_ratio"]

    # 去除出栏总量这一列
    animal = animal.drop(columns=["state_slaughtered_broilers","inventory_ratio"])

    # 保存
    animal.to_csv("动物_ok/"+str(y)+".csv",index=False)

In [91]:
def merge_left(data_left,data_right,item,new_name_item,on="OBJECTID"):
    data_item = data_right.loc[data_right["Data Item"]==item].reset_index(drop=True)
    data_left = data_left.merge(data_item[["Value",on]],on=on,how="left")
    return data_left.rename(columns={"Value":new_name_item}) # 改名

In [61]:
# 补全牲畜的动物缺失（保密）数据
year = ["1997","2002","2007","2012","2017"]
for y in year:
    # Load the datasets
    farm_operations_path = '动物_ok/Farm operations/'+y+'.csv'
    state_animal_path = '动物_ok/livestock州总量/'+y+'.csv'
    county_animal_path = '动物_ok/'+y+'.csv'

    farm_operations_df = pd.read_csv(farm_operations_path)
    state_animal_df = pd.read_csv(state_animal_path)
    county_animal_df = pd.read_csv(county_animal_path)

    # Step 1: Calculate the ratio 'r' for each county
    # Relevant Domain Categories
    relevant_categories = ["NOT SPECIFIED", "AREA OPERATED: (500 TO 999 ACRES)", "AREA OPERATED: (1,000 TO 1,999 ACRES)", "AREA OPERATED: (2,000 OR MORE ACRES)"]

    # Convert 'Value' column to numeric, replacing commas and coercing non-numeric values to NaN
    farm_operations_df['Value'] = pd.to_numeric(farm_operations_df['Value'].str.replace(',', ''), errors='coerce')

    # Filter relevant rows and pivot the table
    farm_ops_filtered = farm_operations_df[farm_operations_df['Domain Category'].isin(relevant_categories)]
    pivot_farm_ops = farm_ops_filtered.pivot_table(index='County', columns='Domain Category', values='Value', aggfunc='sum')

    # Calculate the ratio 'r'
    pivot_farm_ops['r'] = (pivot_farm_ops["AREA OPERATED: (500 TO 999 ACRES)"] + pivot_farm_ops["AREA OPERATED: (1,000 TO 1,999 ACRES)"] + pivot_farm_ops["AREA OPERATED: (2,000 OR MORE ACRES)"]) / pivot_farm_ops["NOT SPECIFIED"]

    # Step 2: Calculate missing inventory for each animal type in each state
    # Converting 'Value' column to numeric in state_animal_df
    # 先将 'Value' 列转换为字符串类型，然后再进行替换操作
    state_animal_df['Value'] = pd.to_numeric(state_animal_df['Value'].astype(str).str.replace(',', ''), errors='coerce')

    # Summarizing state animal data
    state_animal_summary = state_animal_df.pivot_table(index='State', columns='Data Item', values='Value', aggfunc='sum')

    # For sheep, we need to sum 'GOATS - INVENTORY' and 'SHEEP, INCL LAMBS - INVENTORY'
    state_animal_summary['SHEEP_TOTAL'] = state_animal_summary.get('GOATS - INVENTORY', 0) + state_animal_summary.get('SHEEP, INCL LAMBS - INVENTORY', 0)


    # Correctly converting relevant columns in county_animal_df to numeric
    county_animal_cols = ['cattle_cow_beef', 'cattle_cow_milk', 'cattle_excl_cows', 'hogs', 'sheep_goats']
    for col in county_animal_cols:
        if county_animal_df[col].dtype == object:
            county_animal_df[col] = pd.to_numeric(county_animal_df[col].str.replace(',', ''), errors='coerce')

    # Summarizing county animal data again
    county_animal_totals = county_animal_df.groupby('State')[county_animal_cols].sum()

    # Aligning the columns of state_animal_summary with those of county_animal_totals
    state_animal_summary_aligned = state_animal_summary.rename(columns={
        'CATTLE, COWS, BEEF - INVENTORY': 'cattle_cow_beef',
        'CATTLE, COWS, MILK - INVENTORY': 'cattle_cow_milk',
        'CATTLE, (EXCL COWS) - INVENTORY': 'cattle_excl_cows',
        'HOGS - INVENTORY': 'hogs',
        'SHEEP_TOTAL': 'sheep_goats'
    })

    # Check if 'cattle_excl_cows' column exists in state_animal_summary_renamed
    if 'cattle_excl_cows' in state_animal_summary_aligned.columns:
        # If it exists, proceed as before
        missing_inventory = state_animal_summary_aligned[['cattle_cow_beef', 'cattle_cow_milk', 'cattle_excl_cows', 'hogs', 'sheep_goats']].sub(county_animal_totals, fill_value=0)
    else:
        # If it doesn't exist, exclude it from the calculation
        missing_inventory = state_animal_summary_aligned[['cattle_cow_beef', 'cattle_cow_milk', 'hogs', 'sheep_goats']].sub(county_animal_totals, fill_value=0)

    # Step 3: Allocate missing data to counties based on ratio 'r'
    # Identifying counties with missing data in county_animal_df
    missing_data_counties = county_animal_df[county_animal_df.isna().any(axis=1)][['State', 'County']].merge(pivot_farm_ops[['r']], left_on='County', right_index=True, how='left')

    # Calculate sum of 'r' for each state
    sum_r_per_state = missing_data_counties.groupby('State')['r'].sum().reset_index()
    missing_inventory_with_r = missing_inventory.reset_index().merge(sum_r_per_state, on='State', how='left')

    # Allocate missing data to counties
    for animal in ['cattle_cow_beef', 'cattle_cow_milk', 'cattle_excl_cows', 'hogs', 'sheep_goats']:
        missing_data_counties[animal] = missing_data_counties.apply(
            lambda x: (missing_inventory_with_r.loc[missing_inventory_with_r['State'] == x['State'], animal] * 
                    (x['r'] / missing_inventory_with_r.loc[missing_inventory_with_r['State'] == x['State'], 'r'])).values[0] if x['r'] else 0,
            axis=1
        )

    # Step 4: Fill missing data in 'county_animal.csv'
    # Prepare missing data for merging
    missing_data_to_merge = missing_data_counties[['State', 'County', 'cattle_cow_beef', 'cattle_cow_milk', 'cattle_excl_cows', 'hogs', 'sheep_goats']]

    # Merging the missing data with the original county_animal_df
    county_animal_filled = county_animal_df.merge(missing_data_to_merge, on=['State', 'County'], how='left', suffixes=('', '_missing'))

    # Filling the missing values in county_animal_df with the calculated missing data
    for animal in ['cattle_cow_beef', 'cattle_cow_milk', 'cattle_excl_cows', 'hogs', 'sheep_goats']:
        county_animal_filled[animal].fillna(county_animal_filled[f'{animal}_missing'], inplace=True)

    # Dropping the temporary columns used for missing data
    county_animal_filled.drop(columns=[col for col in county_animal_filled if '_missing' in col], inplace=True)

    # save
    county_animal_filled.to_csv("动物_保密_ok/"+y+".csv",index=False)




C:\Users\typing\AppData\Local\Temp\ipykernel_4404\1533158413.py:9: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  farm_operations_df = pd.read_csv(farm_operations_path)
C:\Users\typing\AppData\Local\Temp\ipykernel_4404\1533158413.py:9: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  farm_operations_df = pd.read_csv(farm_operations_path)


In [62]:
# 补全禽类缺失的保密数据
# 补全牲畜的动物缺失（保密）数据
year = ["1997","2002","2007","2012","2017"]
for y in year:
    # Load the datasets
    farm_operations_path = '动物_ok/Farm operations/'+y+'.csv'
    state_animal_path = '动物_ok/poultry州的蛋鸡和肉鸡总量/'+y+'.csv'
    county_animal_path = '动物_保密_ok/'+y+'.csv'

    farm_operations_df = pd.read_csv(farm_operations_path)
    state_animal_df = pd.read_csv(state_animal_path)
    county_animal_df = pd.read_csv(county_animal_path)

    # Step 1: Calculate the ratio 'r' for each county
    # Relevant Domain Categories
    relevant_categories = ["NOT SPECIFIED", "AREA OPERATED: (500 TO 999 ACRES)", "AREA OPERATED: (1,000 TO 1,999 ACRES)", "AREA OPERATED: (2,000 OR MORE ACRES)"]

    # Convert 'Value' column to numeric, replacing commas and coercing non-numeric values to NaN
    farm_operations_df['Value'] = pd.to_numeric(farm_operations_df['Value'].str.replace(',', ''), errors='coerce')

    # Filter relevant rows and pivot the table
    farm_ops_filtered = farm_operations_df[farm_operations_df['Domain Category'].isin(relevant_categories)]
    pivot_farm_ops = farm_ops_filtered.pivot_table(index='County', columns='Domain Category', values='Value', aggfunc='sum')

    # Calculate the ratio 'r'
    pivot_farm_ops['r'] = (pivot_farm_ops["AREA OPERATED: (500 TO 999 ACRES)"] + pivot_farm_ops["AREA OPERATED: (1,000 TO 1,999 ACRES)"] + pivot_farm_ops["AREA OPERATED: (2,000 OR MORE ACRES)"]) / pivot_farm_ops["NOT SPECIFIED"]

    # Step 2: Calculate missing inventory for each animal type in each state
    # Converting 'Value' column to numeric in state_animal_df
    # 先将 'Value' 列转换为字符串类型，然后再进行替换操作
    state_animal_df['Value'] = pd.to_numeric(state_animal_df['Value'].astype(str).str.replace(',', ''), errors='coerce')

    # Summarizing state animal data
    state_animal_summary = state_animal_df.pivot_table(index='State', columns='Data Item', values='Value', aggfunc='sum')

    # For sheep, we need to sum 'GOATS - INVENTORY' and 'SHEEP, INCL LAMBS - INVENTORY'
    # state_animal_summary['SHEEP_TOTAL'] = state_animal_summary.get('GOATS - INVENTORY', 0) + state_animal_summary.get('SHEEP, INCL LAMBS - INVENTORY', 0)


    # Correctly converting relevant columns in county_animal_df to numeric
    county_animal_cols = ["broilers","layers"]
    for col in county_animal_cols:
        if county_animal_df[col].dtype == object:
            county_animal_df[col] = pd.to_numeric(county_animal_df[col].str.replace(',', ''), errors='coerce')

    # Summarizing county animal data again
    county_animal_totals = county_animal_df.groupby('State')[county_animal_cols].sum()

    # Aligning the columns of state_animal_summary with those of county_animal_totals
    state_animal_summary_aligned = state_animal_summary.rename(columns={
        'CHICKENS, BROILERS - INVENTORY': 'broilers',
        'CHICKENS, LAYERS - INVENTORY': 'layers'
    })

    # Check if 'cattle_excl_cows' column exists in state_animal_summary_renamed
    if 'cattle_excl_cows' in state_animal_summary_aligned.columns:
        # If it exists, proceed as before
        missing_inventory = state_animal_summary_aligned[['']].sub(county_animal_totals, fill_value=0)
    else:
        # If it doesn't exist, exclude it from the calculation
        missing_inventory = state_animal_summary_aligned[['broilers','layers']].sub(county_animal_totals, fill_value=0)

    # Step 3: Allocate missing data to counties based on ratio 'r'
    # Identifying counties with missing data in county_animal_df
    missing_data_counties = county_animal_df[county_animal_df.isna().any(axis=1)][['State', 'County']].merge(pivot_farm_ops[['r']], left_on='County', right_index=True, how='left')

    # Calculate sum of 'r' for each state
    sum_r_per_state = missing_data_counties.groupby('State')['r'].sum().reset_index()
    missing_inventory_with_r = missing_inventory.reset_index().merge(sum_r_per_state, on='State', how='left')

    # Allocate missing data to counties
    for animal in ['broilers','layers']:
        missing_data_counties[animal] = missing_data_counties.apply(
            lambda x: (missing_inventory_with_r.loc[missing_inventory_with_r['State'] == x['State'], animal] * 
                    (x['r'] / missing_inventory_with_r.loc[missing_inventory_with_r['State'] == x['State'], 'r'])).values[0] if x['r'] else 0,
            axis=1
        )

    # Step 4: Fill missing data in 'county_animal.csv'
    # Prepare missing data for merging
    missing_data_to_merge = missing_data_counties[['State', 'County', 'broilers','layers']]

    # Merging the missing data with the original county_animal_df
    county_animal_filled = county_animal_df.merge(missing_data_to_merge, on=['State', 'County'], how='left', suffixes=('', '_missing'))

    # Filling the missing values in county_animal_df with the calculated missing data
    for animal in ['broilers','layers']:
        county_animal_filled[animal].fillna(county_animal_filled[f'{animal}_missing'], inplace=True)

    # Dropping the temporary columns used for missing data
    county_animal_filled.drop(columns=[col for col in county_animal_filled if '_missing' in col], inplace=True)

    # save
    county_animal_filled.to_csv("动物_保密_ok/"+y+".csv",index=False)




C:\Users\typing\AppData\Local\Temp\ipykernel_4404\3603674257.py:10: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  farm_operations_df = pd.read_csv(farm_operations_path)
C:\Users\typing\AppData\Local\Temp\ipykernel_4404\3603674257.py:10: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  farm_operations_df = pd.read_csv(farm_operations_path)


In [65]:
# 保密数据补全之后，重新计算出栏数据
# 将这3种羊的Value相加形成新的一列羊
def safe_numeric(x):
    try:
        return pd.to_numeric(x)
    except ValueError:
        return 0
import pandas as pd
year = ["1997","2002","2007","2012","2017"]
for y in year:  
    animal = pd.read_csv("动物_ok/"+y+".csv")
  # 计算肉牛的出栏量
    # 确定每个州的肉牛出栏总量
    state_slaughtered = pd.read_csv("standard/state/动物类/livestocks_standard/"+str(y)+".csv")
    # 取出牛贸易屠宰的，使用左连接
    state_slaughtered_cattle_beef_com = state_slaughtered.loc[state_slaughtered["Data Item"]=="CATTLE, CALVES, SLAUGHTER, COMMERCIAL - SLAUGHTERED, MEASURED IN HEAD"].reset_index(drop=True)
    animal = animal.merge(state_slaughtered_cattle_beef_com[["Value","State"]],on="State",how="left")
    animal = animal.rename(columns={"Value":"state_slaughtered_cattle_beef_com"}) # 改名

    # 取出牛农场屠宰的，使用左连接
    state_slaughtered_cattle_beef_farm = state_slaughtered.loc[state_slaughtered["Data Item"]=="CATTLE, INCL CALVES, SLAUGHTER, ON FARM - SLAUGHTERED, MEASURED IN HEAD"].reset_index(drop=True)
    animal = animal.merge(state_slaughtered_cattle_beef_farm[["Value","State"]],on="State",how="left")
    animal = animal.rename(columns={"Value":"state_slaughtered_cattle_beef_farm"}) # 改名

    # 将这2种牛的Value相加形成新的一列作为出栏总量
    cols_to_convert = ["state_slaughtered_cattle_beef_com","state_slaughtered_cattle_beef_farm"]

    for col in cols_to_convert:
        animal[col] = animal[col].str.replace(',', '')  # remove commas

    animal["cattle_slaughtered_sum"] = animal[cols_to_convert].applymap(safe_numeric).sum(axis=1) # 新的一列

    # 删除这2列
    animal = animal.drop(columns=cols_to_convert)

    # 计算肉牛在该州的存栏占比
    # animal["cattle_cow_beef"] = animal["cattle_cow_beef"].str.replace(',', '')  # remove commas
    # 将该列转化成数值类型，转化不了的用0代替
    # animal["cattle_cow_beef"] = pd.to_numeric(animal["cattle_cow_beef"], errors='coerce').fillna(0)
    # 计算每个州的肉牛存栏总数
    state_total = animal.groupby('State')['cattle_cow_beef'].transform('sum')
    # 计算每个县在其所在州的肉牛存栏占比
    animal["inventory_ratio"] = animal["cattle_cow_beef"] / state_total

    # 用该县所在州的出栏总和乘以该县的存栏占比，得到该县的肉牛出栏量
    animal["cattle_beef_slaughtered"] = animal["cattle_slaughtered_sum"] * animal["inventory_ratio"]

    # 去除出栏总量这一列
    animal = animal.drop(columns=["cattle_slaughtered_sum","inventory_ratio"])

    # 同样的方法计算猪的出栏量
    # 确定每个州的猪出栏总量
    # 取出猪贸易屠宰的，使用左连接
    state_slaughtered_hogs_com = state_slaughtered.loc[state_slaughtered["Data Item"]=="HOGS, SLAUGHTER, COMMERCIAL - SLAUGHTERED, MEASURED IN HEAD"].reset_index(drop=True)
    animal = animal.merge(state_slaughtered_hogs_com[["Value","State"]],on="State",how="left")
    animal = animal.rename(columns={"Value":"state_slaughtered_hogs_com"}) # 改名

    # 取出猪农场屠宰的，使用左连接
    state_slaughtered_hogs_farm = state_slaughtered.loc[state_slaughtered["Data Item"]=="HOGS, SLAUGHTER, ON FARM - SLAUGHTERED, MEASURED IN HEAD"].reset_index(drop=True)
    animal = animal.merge(state_slaughtered_hogs_farm[["Value","State"]],on="State",how="left")
    animal = animal.rename(columns={"Value":"state_slaughtered_hogs_farm"}) # 改名

    # 将这2种猪的Value相加形成新的一列作为出栏总量
    cols_to_convert = ["state_slaughtered_hogs_com","state_slaughtered_hogs_farm"]

    for col in cols_to_convert:
        animal[col] = animal[col].str.replace(',', '')  # remove commas

    animal["hogs_slaughtered_sum"] = animal[cols_to_convert].applymap(safe_numeric).sum(axis=1) # 新的一列

    # 删除这2列
    animal = animal.drop(columns=cols_to_convert)

    # 计算猪在该州的存栏占比
    # animal["hogs"] = animal["hogs"].str.replace(',', '')  # remove commas
    # 将该列转化成数值类型，转化不了的用0代替
    # animal["hogs"] = pd.to_numeric(animal["hogs"], errors='coerce').fillna(0)
    # 计算每个州的猪存栏总数
    state_total = animal.groupby('State')['hogs'].transform('sum')
    # 计算每个县在其所在州的猪存栏占比
    animal["inventory_ratio"] = animal["hogs"] / state_total

    # 用该县所在州的出栏总和乘以该县的存栏占比，得到该县的猪出栏量
    animal["hogs_slaughtered"] = animal["hogs_slaughtered_sum"] * animal["inventory_ratio"]

    # 去除出栏总量这一列
    animal = animal.drop(columns=["hogs_slaughtered_sum","inventory_ratio"])

    # 接下来计算肉鸡出栏量
    # 确定每个州的肉鸡出栏总量
    state_slaughtered = pd.read_csv("standard/state/动物类/poultry_standard/"+str(y)+".csv")

    # 取出肉鸡屠宰的，使用左连接
    state_slaughtered_broilers = state_slaughtered.loc[state_slaughtered["Data Item"]=="CHICKENS, YOUNG, SLAUGHTER, FI - SLAUGHTERED, MEASURED IN HEAD"].reset_index(drop=True)
    animal = animal.merge(state_slaughtered_broilers[["Value","State"]],on="State",how="left")
    animal = animal.rename(columns={"Value":"state_slaughtered_broilers"}) # 改名

    # 将该列转化成数值类型，转化不了的用0代替
    animal["state_slaughtered_broilers"] = animal["state_slaughtered_broilers"].str.replace(',', '')  # remove commas
    animal["state_slaughtered_broilers"] = pd.to_numeric(animal["state_slaughtered_broilers"], errors='coerce').fillna(0)

    # 将该列转化成数值类型，转化不了的用0代替
    # animal["broilers"] = animal["broilers"].str.replace(',', '')  # remove commas
    # animal["broilers"] = pd.to_numeric(animal["broilers"], errors='coerce').fillna(0)

    # 计算每个州的肉鸡存栏总数
    state_total = animal.groupby('State')['broilers'].transform('sum')
    # 计算每个县在其所在州的肉鸡存栏占比
    animal["inventory_ratio"] = animal["broilers"] / state_total

    # 用该县所在州的出栏总和乘以该县的存栏占比，得到该县的肉鸡出栏量
    animal["broilers_slaughtered"] = animal["state_slaughtered_broilers"] * animal["inventory_ratio"]

    # 去除出栏总量这一列
    animal = animal.drop(columns=["state_slaughtered_broilers","inventory_ratio"])

    # 保存
    animal.to_csv("动物_ok_tmp/"+str(y)+".csv",index=False)

---
# 校对数据
动物的单位是统一的，都是An，
农作物的单位要统一成英亩或者公顷


In [10]:
# 与FAO数据校对,注意单位
# 美国、巴西,单个国家的校对
def single_proofread(data,data_fao,data_fao_item="item"):
    # data是之前已经整理好的数据，现在要被校对
    # data_fao是fao上的国家总量
    data['标记'] = ""

    # 首先筛选出动物种类
    animal_species = data_fao[data_fao_item].unique() 
    value_name = "Value" if "Value" in data_fao.columns else "value"

    for animal in animal_species:
        
        # 首先判断这个动物种类是否在data里面
        if animal in data.columns:
            # 首先转化列里面数值保证能进行四则运算
            data[animal] = data[animal].astype(str)
            data[animal] = data[animal].str.replace(',', '')  # remove commas
            data[animal] = pd.to_numeric(data[animal], errors='coerce')
            data_fao[value_name] = data_fao[value_name].astype(str)
            data_fao[value_name] = data_fao[value_name].str.replace(',', '')
            data_fao[value_name] = pd.to_numeric(data_fao[value_name], errors='coerce')

            # 首先计算data里面每个县占比
            proportions = data[animal] / data[animal].sum()

            # 如果FAO数据缺失那么就不进行校对
            value = data_fao[data_fao[data_fao_item]==animal][value_name].values[0]

            if abs(value-data[animal].sum()) > 0.2*max(value,data[animal].sum()):
                # fao总量与国家总量差别太大的不要
                data["标记"] += "种类："+animal+","+"FAO总量："+str(value)+","+"国家总量："+str(data[animal].sum())+"; "

                # continue 

            # 检查value是否为数值类型
            if isinstance(value, (int, float)):
                # 如果是数值类型，检查是否大于等于0
                if value > 0:
                    data[animal] = proportions*value
    return data


In [1]:
# 规范化fao数据，注意单位
# 输入所有年份的fao原始数据,输出分年份规范化好的fao数据
def fao_standard(fao_data,params,target_path):
    # params是字典，键是fao数据的动物种类名称，值是键对应的要转化的种类名称，所有值是列表则证明要将这两项相加
    # 确保数值列可以做四则运算
    value_name = "Value" if "Value" in fao_data.columns else "value"
    fao_data[value_name] = fao_data[value_name].astype(str)
    fao_data[value_name] = fao_data[value_name].str.replace(',', '')
    fao_data[value_name] = pd.to_numeric(fao_data[value_name], errors='coerce')

    # 单位换算，1英亩等于0.404686公顷，农作物面积的时候才需要
    # fao_data[value_name] = fao_data[value_name] * 0.404686

    for year,group in fao_data.groupby("Year"):
        # 一年一年来，搞完保存

        # fao_data数据只有Item列需要,用replace替换
        for item in params:
            if isinstance(item,str):
                group["Item"] = group['Item'].replace(item,params[item])
            elif isinstance(item,tuple):
                # 先相加，按照年份，然后形成新的一个值
                for i in range(1,len(item)):
                    group.loc[group["Item"]==item[0],value_name] += group[group['Item']==item[i]][value_name].values[0]
                    
                group["Item"] = group['Item'].replace(item[0],params[item]) 
            else:
                print("params输入格式错误,错误的键为{}".format(item))

        # 保存
        group.to_csv(target_path+str(year)+".csv")

In [9]:
params = {
    "Raw milk of cattle":"cattle_cow_milk",
    ("Meat of cattle with the bone, fresh or chilled","Meat of buffalo, fresh or chilled"):"cattle_cow_beef",
    # "excl cattle":
    "Meat of pig with the bone, fresh or chilled":"hogs",
    ("Sheep","Goats"):"sheep_goats",
    "Hen eggs in shell, fresh":"layers",
    "Meat of chickens, fresh or chilled":"broilers"
}


In [ ]:

fao_standard()

In [12]:
import pandas as pd 
# 与FAO数据校对
year = ["1997","2002","2007","2012","2017"]
for y in year:
    data = pd.read_csv("D:/中科院数据下载/废弃数据/动物_ok/"+y+".csv")
    data_fao = pd.read_csv("D:/中科院数据下载/废弃数据/动物_ok/fao数据/"+y+".csv")
    data_ok = single_proofread(data,data_fao)
    data_ok.to_csv("D:/中科院数据下载/USDAquickstats/动物_fao_ok/"+y+".csv",index=False,encoding="utf-8-sig")

In [7]:
data_ok

,OBJECTID,State,County,year,cattle_cow_beef,cattle_cow_milk,cattle_excl_cows,hogs,sheep_goats,broilers,layers,horses,donkeys,mules,rabbits,cattle_beef_slaughtered,hogs_slaughtered,broilers_slaughtered,标记
0,26,ALABAMA,COLBERT,2017,9020.0,50.879503,6789.0,150.799554,1195.068885,1094358.0,NaN,NaN,NaN,NaN,NaN,43.711369,185.117459,5.768382e+06,"种类：layers,FAO总量：381175000.0,国家总量：170230374.0; ..."
1,39,ALABAMA,FRANKLIN,2017,15552.0,28.035645,10585.0,29.929682,2239.235004,6550273.0,120431.0,NaN,NaN,NaN,NaN,75.365767,36.740870,3.452662e+07,"种类：layers,FAO总量：381175000.0,国家总量：170230374.0; ..."
2,48,ALABAMA,LAUDERDALE,2017,16372.0,33.227431,13920.0,626.221048,3303.656528,915548.0,204921.0,NaN,NaN,NaN,NaN,79.339527,768.732045,4.825871e+06,"种类：layers,FAO总量：381175000.0,国家总量：170230374.0; ..."
3,49,ALABAMA,LAWRENCE,2017,16607.0,85.145291,14498.0,762.055761,1748.041437,7882206.0,214571.0,NaN,NaN,NaN,NaN,80.478349,935.479069,4.154726e+07,"种类：layers,FAO总量：381175000.0,国家总量：170230374.0; ..."
4,51,ALABAMA,LIMESTONE,2017,0.0,NaN,9233.0,2542.871867,3415.061254,2038192.0,68262.0,NaN,NaN,NaN,NaN,0.000000,3121.560821,1.074335e+07,"种类：layers,FAO总量：381175000.0,国家总量：170230374.0; ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3061,2790,TEXAS,ZAVALA,2017,12865.0,NaN,37480.0,0.000000,1119.111117,0.0,414.0,NaN,NaN,NaN,NaN,34.373095,0.000000,0.000000e+00,"种类：layers,FAO总量：381175000.0,国家总量：170230374.0; ..."
3062,2567,TEXAS,CAMERON,2017,8893.0,NaN,4508.0,925.517873,3597.359898,1171.0,7021.0,NaN,NaN,NaN,NaN,23.760586,1691.783262,6.912908e+03,"种类：layers,FAO总量：381175000.0,国家总量：170230374.0; ..."
3063,2644,TEXAS,HIDALGO,2017,17058.0,NaN,13229.0,1282.371779,11605.334197,1947.0,12643.0,NaN,NaN,NaN,NaN,45.576079,2344.087754,1.149396e+04,"种类：layers,FAO总量：381175000.0,国家总量：170230374.0; ..."
3064,2750,TEXAS,STARR,2017,0.0,NaN,26037.0,506.502318,6291.328740,870.0,4346.0,NaN,NaN,NaN,NaN,0.000000,925.851536,5.135977e+03,"种类：layers,FAO总量：381175000.0,国家总量：170230374.0; ..."
